# 處理美股資料
    ["AAPL", "apple"],
    ["GOOGL", "google"],
    ["MSFT", "microsoft"],
    ["AMZN", "amazon"],
    ["TSLA", "tesla"],
    ["FB", "facebook"],
    ["BRK.B", "berkshire hathaway"],
    ["JPM", "jpmorgan chase"],
    ["JNJ", "johnson & johnson"],
    ["V", "visa"],
    ["WMT", "walmart"],
    ["PG", "procter & gamble"],
    ["MA", "mastercard"],
    ["UNH", "unitedhealth group"],
    ["NVDA", "nvidia"],
    ["HD", "home depot"],
    ["DIS", "disney"],
    ["BAC", "bank of america"],
    ["PYPL", "paypal"],
    ["ADBE", "adobe"],
    ["CMCSA", "comcast"],
    ["KO", "coca-cola"],
    ["INTC", "intel"],
    ["NFLX", "netflix"],
    ["PEP", "pepsico"],
    ["T", "at&t"],
    ["ABT", "abbott laboratories"],
    ["CRM", "salesforce"],
    ["CSCO", "cisco"],
    ["ABBV", "abbvie"],
    ["XOM", "exxonmobil"],
    ["NKE", "nike"],
    ["AVGO", "broadcom"],
    ["TMO", "thermo fisher scientific"],
    ["CVX", "chevron"],
    ["QCOM", "qualcomm"],
    ["LLY", "eli lilly"],
    ["DHR", "danaher"],
    ["NEE", "nextera energy"],
    ["ACN", "accenture"],
    ["TXN", "texas instruments"],
    ["COST", "costco"],
    ["UNP", "union pacific"],
    ["LIN", "linde"],
    ["PM", "philip morris international"],
    ["ORCL", "oracle"],
    ["HON", "honeywell"],

In [2]:
import csv
import os
from datetime import datetime

# 讀取舊格式的檔案並轉換為新格式
def convert_file(input_file, output_file):
    with open(input_file, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)
        rows = []

        # 逐行處理資料
        for row in reader:
            # 處理日期格式
            date = datetime.strptime(row["Date"], "%m/%d/%Y").strftime("%Y%m%d")

            # 去除金額前的 $ 符號
            close_price = row["Close/Last"].replace("$", "")
            open_price = row["Open"].replace("$", "")
            high_price = row["High"].replace("$", "")
            low_price = row["Low"].replace("$", "")

            # 移除 Volume 的逗號
            volume = row["Volume"].replace(",", "")

            rows.append({
                "Date": date,
                "Volume": volume,
                "Open": open_price,
                "High": high_price,
                "Low": low_price,
                "Close": close_price,
            })

    # 將資料寫入新的檔案
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, mode='w', encoding='utf-8', newline='') as outfile:
        fieldnames = ["Date", "Volume",  "Open", "High", "Low", "Close"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)

        writer.writeheader()
        writer.writerows(rows)

# 指定檔案路徑
ticker = "NVDA"
input_file = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'us', 'stock_price_raw', ticker + '.csv')
output_file = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'us', 'stock_price', ticker + '.csv')

# 執行轉換
convert_file(input_file, output_file)
print(f"檔案已轉換並儲存至 {output_file}")

檔案已轉換並儲存至 /Users/yanyifu/Documents/_Coding/LLM Predict Stock/history_data/us/stock_price/NVDA.csv


# 畫技術分析


In [3]:
import pandas as pd
import numpy as np
import json
import requests
import time
import os


def cal_tech(id):

    path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
    # folder_list = ['2330', '2454', '2317', '2382', '2308', '3711']


    stock_tech = pd.read_csv(os.path.join(path , id + 'twse.csv'))

    # # 將stock_tech的值都換成float
    # stock_tech['Open'] = stock_tech['Open'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Close'] = stock_tech['Close'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['High'] = stock_tech['High'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Low'] = stock_tech['Low'].map(lambda x: float(x.replace(',', '')))


    # 計算移動均線 (SMA)
    stock_tech['MA5'] = stock_tech['Close'].rolling(window=5).mean()
    stock_tech['MA10'] = stock_tech['Close'].rolling(window=10).mean()
    stock_tech['MA20'] = stock_tech['Close'].rolling(window=20).mean()

    # 計算指數移動平均線 (EMA)
    stock_tech['EMA5'] = stock_tech['Close'].ewm(span=5, adjust=False).mean()
    stock_tech['EMA12'] = stock_tech['Close'].ewm(span=12, adjust=False).mean()
    stock_tech['EMA26'] = stock_tech['Close'].ewm(span=26, adjust=False).mean()

    # 計算MACD和DIF
    stock_tech['DIF'] = stock_tech['EMA12'] - stock_tech['EMA26']
    stock_tech['MACD'] = stock_tech['DIF'].ewm(span=9, adjust=False).mean()


    # 計算RSI
    def calculate_rsi(series, period=14):
        delta = series.diff(1)
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    stock_tech['RSI'] = calculate_rsi(stock_tech['Close'])

    # 計算KDJ
    low_list = stock_tech['Low'].rolling(9, min_periods=9).min()
    low_list.fillna(value=stock_tech['Low'].expanding().min(), inplace=True)
    high_list = stock_tech['High'].rolling(9, min_periods=9).max()
    high_list.fillna(value=stock_tech['High'].expanding().max(), inplace=True)
    rsv = (stock_tech['Close'] - low_list) / (high_list - low_list) * 100


    # 初始值設定
    stock_tech['K'] = 50
    stock_tech['D'] = 50

    for i in range(1, len(stock_tech)):
        stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
        stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


    # 顯示DataFrame
    stock_tech.to_csv(os.path.join(path , id + 'tech.csv'), encoding='utf-8', index=False)

In [5]:
from io import StringIO

path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
folder_list = ['2330']

for id in folder_list:
    try:
        stock_tech = fetch_data_twse(id, 2022, 2024) # 這裡可以自行設定要抓取的時間範圍
        # Remove commas and convert to numeric types
        df = stock_tech.applymap(lambda x: pd.to_numeric(str(x).replace(",", "").replace('"', ''), errors='coerce'))
        df.to_csv(os.path.join(path , id + 'twse.csv')) # 將資料存成 csv 檔
        print('save ' + id + 'twse.csv')
        cal_tech(id)
        print('save ' + id + 'tech.csv')
    except Exception as e:
        print("Error: " + id + " " + str(e))
        pass

Fetching data for 2330 from 2022 to 2024...
Data not found for 202411
save 2330twse.csv
save 2330tech.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_44223/127480904.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = stock_tech.applymap(lambda x: pd.to_numeric(str(x).replace(",", "").replace('"', ''), errors='coerce'))
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_44223/2247075068.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '66.66666666666666' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_44223/2247075068.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '55.55555555555554' has dtype incompatible with int64, please explicitly cast to a 